In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.ensemble import RandomForestClassifier

import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
Q1_path=os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(Q1_path)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
#Plot Target Distribution
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df_d=df.drop('Order_ID',axis=1)

In [ ]:
df_d.head()    # to check if its really dropped or not

In [ ]:
df_d.isnull().sum()

In [ ]:
df_d=df_d.fillna(df.mean(numeric_only=True))  # Fill Any Numeric Column with mean and fill categorial ones with mode
categorical_cols = df_d.select_dtypes(include=["object"]).columns
# For loop to fill any Categorial Column with the majority.
print("Categorical Columns:", list(categorical_cols))
df_d['Traffic_Level'] = df_d['Traffic_Level'].fillna(df_d['Traffic_Level'].mode()[0])
for col in categorical_cols:
  df_d[col] = df_d[col].fillna(df_d[col].mode()[0])

df_d.isnull().sum()

In [ ]:
df_d.isnull().sum()



In [ ]:
df_d.head()

In [ ]:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_d)

In [ ]:
for col in categorical_cols:
    le = LabelEncoder()
    df_d[col] = le.fit_transform(df_d[col].astype(str))

df_d.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_d.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_d[numerical_cols] = scaler.fit_transform(df_d[numerical_cols])
df_d.head()

In [ ]:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_d, "Delivery_Time")


In [ ]:
X=df_d.drop("Delivery_Time",axis=1)
y=df_d["Delivery_Time"]

In [ ]:
X

In [ ]:
y

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split
n_splits = 5  # K=5 Folds
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model=RandomForestRegressor(n_estimators=100)
  model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
mae=mean_absolute_error(y_test,y_pred)
mae


In [ ]:
sklearn_models = {

   "Random Forest": RandomForestRegressor(n_estimators=100)
}

In [ ]:
importance = list(zip(X.columns, model.feature_importances_))
sorted_importance = sorted(importance, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_importance)

# Plot feature importances
plt.figure(figsize=(18, 14))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('Random Forest Importance Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()

In [ ]:
plt.hist(y_pred, bins=30, edgecolor='black')
plt.title('Model Prediction')
plt.show()

In [ ]:
# Task Bonus: Write your code here: